In [1]:
import numpy as np
import struct

with open('thick_378786.raw', 'rb') as f:
    # Чтение заголовка
    x = struct.unpack('>i', f.read(4))[0]      # big-endian int
    y = struct.unpack('>i', f.read(4))[0]
    channels = struct.unpack('>i', f.read(4))[0]
    ranges = struct.unpack('>i', f.read(4))[0]
    xSize = struct.unpack('>d', f.read(8))[0]  # big-endian double
    ySize = struct.unpack('>d', f.read(8))[0]
    min_val = struct.unpack('>d', f.read(8))[0]
    max_val = struct.unpack('>d', f.read(8))[0]
    
    print(f"Размеры: {x}×{y}×{channels}×{ranges}")
    
    # Чтение данных как массива short
    data = np.frombuffer(f.read(), dtype='>i2')  # big-endian short
    data = data.reshape((x, y, channels, ranges))
    
    print(f"Пример значения: {data[0,0,0,0]}")

Размеры: 148×8×4×40
Пример значения: 0


In [7]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import struct
import os

def read_raw_file(filepath):
    """Чтение бинарного файла, созданного Java DataOutputStream"""
    with open(filepath, 'rb') as f:
        # Чтение заголовка (Java использует big-endian)
        x = struct.unpack('>i', f.read(4))[0]           # counts.length
        y = struct.unpack('>i', f.read(4))[0]           # counts[0].length
        channels = struct.unpack('>i', f.read(4))[0]    # counts[0][0].length
        ranges = struct.unpack('>i', f.read(4))[0]      # counts[0][0][0].length
        
        x_size = struct.unpack('>d', f.read(8))[0]
        y_size = struct.unpack('>d', f.read(8))[0]
        min_val = struct.unpack('>d', f.read(8))[0]
        max_val = struct.unpack('>d', f.read(8))[0]
        
        print(f"📊 Заголовок: {x}×{y}×{channels}×{ranges}")
        print(f"📐 Размеры: xSize={x_size}, ySize={y_size}")
        print(f"📈 Диапазон значений: [{min_val}, {max_val}]")
        
        # Чтение данных: short (2 байта, big-endian)
        total_elements = x * y * channels * ranges
        data_flat = np.frombuffer(f.read(2 * total_elements), dtype='>i2')
        
        # Формирование 4D массива [X][Y][Channels][Ranges]
        data = data_flat.reshape((x, y, channels, ranges))
        
        return {
            'x': x, 'y': y, 'channels': channels, 'ranges': ranges,
            'x_size': x_size, 'y_size': y_size,
            'min': min_val, 'max': max_val,
            'data': data
        }


def create_custom_cmap(defectRange):
    """Создание кастомной цветовой карты: 0-4 → зелёный, 5-9 → красный"""
    # 10 цветов для 10 диапазонов
    colors = []
    
    # Диапазоны 0-4: зелёный спектр (от светло-зелёного к тёмному)
    for i in range(defectRange):
        intensity = 0.4 + 0.6 * (i / (defectRange-1))  # от 0.4 до 1.0
        colors.append((0, intensity, 0))  # (R, G, B)
    
    # Диапазоны 5-9: красный спектр (от светло-красного к тёмному)
    for i in range(10-defectRange):
        intensity = 0.4 + 0.6 * (i /(10-defectRange)-1)  # от 0.4 до 1.0
        colors.append((intensity, 0, 0))  # (R, G, B)
    
    return mcolors.ListedColormap(colors)


def visualize_channel(data_2d, channel_idx, output_path=None, defectRange=7):
    """
    Визуализация одного канала.
    data_2d: массив [X][Y][Ranges] - гистограммы для каждой ячейки
    """
    x, y, ranges = data_2d.shape
    
    # Находим индекс диапазона с максимальным количеством попаданий
    dominant_range = np.argmax(data_2d, axis=2)  # [X][Y]
    
    # Создаём кастомную цветовую карту
    cmap = create_custom_cmap(defectRange)
    norm = mcolors.BoundaryNorm(np.arange(11) - 0.5, cmap.N)
    
    # Построение изображения
    plt.figure(figsize=(8, 6))
    # transpose для корректной ориентации (Y по вертикали)
    im = plt.imshow(dominant_range.T, cmap=cmap, norm=norm, 
                    origin='lower', interpolation='nearest',
                    extent=[0, x, 0, y])
    
    plt.title(f'Канал #{channel_idx}\n(цвет = доминирующий диапазон гистограммы)')
    plt.xlabel('X (ячейки)')
    plt.ylabel('Y (ячейки)')
    plt.colorbar(im, ticks=range(10), label='Диапазон значений')
    
    # Подписи для цветовой легенды
    cbar = plt.colorbar(im, ticks=range(10))
    cbar.ax.set_yticklabels([
        '<0.5', '0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9',  # зелёные
        '0.9-1.0', '1.0-1.1', '1.1-1.2', '1.2-1.3', '≥1.3'   # красные
    ], fontsize=8)
    
    plt.grid(False)
    plt.tight_layout()
    
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"💾 Сохранено: {output_path}")
    else:
        plt.show()
    
    plt.close()


def visualize_all_channels(raw_data, output_dir='./output'):
    """Визуализация всех каналов с сохранением результатов"""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    data = raw_data['data']  # [X][Y][Channels][Ranges]
    x, y, channels, ranges = data.shape
    
    print(f"🎨 Генерация визуализаций для {channels} каналов...")
    
    for ch in range(channels):
        # Извлекаем данные для одного канала: [X][Y][Ranges]
        channel_data = data[:, :, ch, :]
        
        output_path = os.path.join(output_dir, f'channel_{ch:03d}.png')
        visualize_channel(channel_data, ch, output_path, 8)
    
    print("✅ Готово!")


# 🔧 Пример использования
if __name__ == '__main__':
    # Путь к вашему файлу
    filepath = './thicks_export/thick_378763.raw'
    
    if not os.path.exists(filepath):
        print(f"❌ Файл не найден: {filepath}")
    else:
        # Чтение данных
        raw_data = read_raw_file(filepath)
        
        # Визуализация всех каналов
        visualize_all_channels(raw_data, output_dir='./visualization_results')
        
        # 🔍 Дополнительно: статистика по доминирующим диапазонам
        data = raw_data['data']
        dominant = np.argmax(data, axis=3)  # [X][Y][Channels]
        
        print("\n📊 Статистика доминирующих диапазонов:")
        for ch in range(raw_data['channels']):
            unique, counts = np.unique(dominant[:, :, ch], return_counts=True)
            total = np.sum(counts)
            print(f"Канал {ch}:")
            for rng, cnt in zip(unique, counts):
                pct = 100 * cnt / total
                status = "🟢 НОРМА" if rng < 5 else "🔴 ДЕФЕКТ"
                print(f"  Диапазон {rng:2d} ({pct:5.1f}%): {cnt:6d} ячеек {status}")

📊 Заголовок: 174×8×4×40
📐 Размеры: xSize=8.700000000000001, ySize=360.0
📈 Диапазон значений: [2.0, 6.0]
🎨 Генерация визуализаций для 4 каналов...
💾 Сохранено: ./visualization_results\channel_000.png
💾 Сохранено: ./visualization_results\channel_001.png
💾 Сохранено: ./visualization_results\channel_002.png
💾 Сохранено: ./visualization_results\channel_003.png
✅ Готово!

📊 Статистика доминирующих диапазонов:
Канал 0:
  Диапазон  0 ( 83.4%):   1161 ячеек 🟢 НОРМА
  Диапазон 31 (  0.2%):      3 ячеек 🔴 ДЕФЕКТ
  Диапазон 32 (  1.8%):     25 ячеек 🔴 ДЕФЕКТ
  Диапазон 33 (  2.0%):     28 ячеек 🔴 ДЕФЕКТ
  Диапазон 34 (  4.2%):     59 ячеек 🔴 ДЕФЕКТ
  Диапазон 35 (  4.1%):     57 ячеек 🔴 ДЕФЕКТ
  Диапазон 36 (  4.2%):     59 ячеек 🔴 ДЕФЕКТ
Канал 1:
  Диапазон  0 ( 82.3%):   1146 ячеек 🟢 НОРМА
  Диапазон 18 (  0.1%):      1 ячеек 🔴 ДЕФЕКТ
  Диапазон 20 (  0.1%):      1 ячеек 🔴 ДЕФЕКТ
  Диапазон 30 (  0.1%):      2 ячеек 🔴 ДЕФЕКТ
  Диапазон 31 (  0.1%):      1 ячеек 🔴 ДЕФЕКТ
  Диапазон 32 (  0.5%):  